## Cell 1 — Imports and Config

In [16]:
import os, json, time, math
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from transformers import AutoModel
import faiss

DATA_PATH = r'C:\visual_searh_project\data\raw_data\Stanford_Online_Products\Stanford_Online_Products'
CKPT_DIR  = r'C:\visual_searh_project\checkpoints'
INDEX_DIR = r'C:\visual_searh_project\faiss_index'
os.makedirs(INDEX_DIR, exist_ok=True)

CKPT_AF    = os.path.join(CKPT_DIR, 'vit_sop_arcface_best.pt')
EMBED_DIM  = 512
MODEL_NAME = 'google/vit-base-patch16-224-in21k'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print(f'FAISS version: {faiss.__version__}')

TEST_TF = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
print('Config ready.')

Device: cuda
GPU: NVIDIA GeForce RTX 3050 Laptop GPU | VRAM: 4.3 GB
FAISS version: 1.14.1
Config ready.


## Cell 2 — Dataset Loading (same as Phase 3)

In [17]:
def parse_sop_annotation(ann_file, data_path):
    samples = []
    with open(ann_file, 'r') as f:
        next(f)
        for line in f:
            parts = line.strip().split()
            if len(parts) < 4: continue
            samples.append((os.path.join(data_path, parts[3]), int(parts[1])))
    return samples

class SOPDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, cid = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, cid, idx

train_samples = parse_sop_annotation(os.path.join(DATA_PATH, 'Ebay_train.txt'), DATA_PATH)
test_samples  = parse_sop_annotation(os.path.join(DATA_PATH, 'Ebay_test.txt'),  DATA_PATH)

# Phase 4 builds the searchable catalog from the FULL dataset (train + test)
# This simulates a real product catalog where every image is searchable
catalog_samples = train_samples + test_samples
catalog_dataset  = SOPDataset(catalog_samples, TEST_TF)
test_dataset     = SOPDataset(test_samples, TEST_TF)

catalog_loader = DataLoader(catalog_dataset, batch_size=64, shuffle=False,
                             num_workers=0, pin_memory=(DEVICE.type=='cuda'))
test_loader    = DataLoader(test_dataset, batch_size=64, shuffle=False,
                             num_workers=0, pin_memory=(DEVICE.type=='cuda'))

print(f'Catalog (train+test): {len(catalog_dataset):,} images')
print(f'Test set (queries)  : {len(test_dataset):,} images')

Catalog (train+test): 120,053 images
Test set (queries)  : 60,502 images


## Cell 3 — Load Fine-tuned ViT-B/16 + ArcFace Model (Phase 3 checkpoint)

In [18]:
ck = torch.load(CKPT_AF, map_location=DEVICE)
print("proj_state keys:")
for k, v in ck['proj_state'].items():
    print(f"  {k}: {v.shape}")

proj_state keys:
  0.weight: torch.Size([768, 768])
  0.bias: torch.Size([768])
  2.weight: torch.Size([512, 768])
  2.bias: torch.Size([512])


In [19]:
class ViTEmbedder(nn.Module):
    def __init__(self, model_name=MODEL_NAME, embed_dim=EMBED_DIM):
        super().__init__()
        self.vit = AutoModel.from_pretrained(model_name)
        self.proj = nn.Sequential(
            nn.Linear(768, 768),   # index 0
            nn.GELU(),             # index 1 (no params)
            nn.Linear(768, embed_dim)  # index 2
        )
    def forward(self, x):
        out = self.vit(pixel_values=x).last_hidden_state[:, 0, :]
        emb = self.proj(out)
        return nn.functional.normalize(emb, p=2, dim=1)

model = ViTEmbedder().to(DEVICE)

print(f'Loading checkpoint: {CKPT_AF}')
ck = torch.load(CKPT_AF, map_location=DEVICE)
model.vit.load_state_dict(ck['vit_state'])
model.proj.load_state_dict(ck['proj_state'])
model.eval()
print(f'Loaded epoch {ck["epoch"]} | training-time val Recall@10 = {ck["recall10"]*100:.2f}%')
print('Model ready for embedding extraction.')

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Loading checkpoint: C:\visual_searh_project\checkpoints\vit_sop_arcface_best.pt
Loaded epoch 10 | training-time val Recall@10 = 84.06%
Model ready for embedding extraction.


## Cell 4 — Extract Embeddings for Full Catalog

In [20]:
def extract_embeddings(loader, model, device, desc='Encoding'):
    all_embs, all_labels, all_idxs = [], [], []
    t0 = time.time()
    with torch.no_grad():
        for i, (imgs, labels, idxs) in enumerate(loader):
            imgs = imgs.to(device)
            with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
                embs = model(imgs)
            all_embs.append(embs.cpu().numpy().astype('float32'))
            all_labels.append(labels.numpy())
            all_idxs.append(idxs.numpy())
            if (i + 1) % 100 == 0:
                print(f'  {desc}: {(i+1)*loader.batch_size:,}/{len(loader.dataset):,}...')
    elapsed = time.time() - t0
    print(f'{desc} complete in {elapsed/60:.1f} min')
    return (np.vstack(all_embs),
            np.concatenate(all_labels),
            np.concatenate(all_idxs))

print('Extracting embeddings for the full catalog (train + test)...')
catalog_embeddings, catalog_labels, catalog_idxs = extract_embeddings(
    catalog_loader, model, DEVICE, desc='Catalog'
)

print(f'\nCatalog embeddings shape: {catalog_embeddings.shape}')
print(f'Embedding dtype: {catalog_embeddings.dtype}')

# Save to disk so we never need to re-run the GPU forward pass again
np.save(os.path.join(INDEX_DIR, 'catalog_embeddings.npy'), catalog_embeddings)
np.save(os.path.join(INDEX_DIR, 'catalog_labels.npy'),     catalog_labels)
np.save(os.path.join(INDEX_DIR, 'catalog_paths.npy'),
        np.array([catalog_samples[i][0] for i in range(len(catalog_samples))]))
print('Saved catalog embeddings, labels, and paths to disk.')

Extracting embeddings for the full catalog (train + test)...
  Catalog: 6,400/120,053...
  Catalog: 12,800/120,053...
  Catalog: 19,200/120,053...
  Catalog: 25,600/120,053...
  Catalog: 32,000/120,053...
  Catalog: 38,400/120,053...
  Catalog: 44,800/120,053...
  Catalog: 51,200/120,053...
  Catalog: 57,600/120,053...
  Catalog: 64,000/120,053...
  Catalog: 70,400/120,053...
  Catalog: 76,800/120,053...
  Catalog: 83,200/120,053...
  Catalog: 89,600/120,053...
  Catalog: 96,000/120,053...
  Catalog: 102,400/120,053...
  Catalog: 108,800/120,053...
  Catalog: 115,200/120,053...
Catalog complete in 52.6 min

Catalog embeddings shape: (120053, 512)
Embedding dtype: float32
Saved catalog embeddings, labels, and paths to disk.


## Cell 5 — Build FAISS Index (IVF + Flat comparison)

In [21]:
# Two indexes are built for comparison:
#   1. IndexFlatIP  — exact brute-force search (ground truth, for accuracy comparison)
#   2. IndexIVFFlat — approximate nearest neighbor search (fast, for deployment)
#
# Embeddings are L2-normalized (done in the model's forward pass), so inner
# product (IP) is equivalent to cosine similarity. This is faster than L2
# distance in FAISS and gives identical ranking for normalized vectors.

d = catalog_embeddings.shape[1]
n = catalog_embeddings.shape[0]
print(f'Building FAISS indexes for {n:,} vectors of dimension {d}')

# 1. Flat index — exact search, used as ground truth for recall comparison
index_flat = faiss.IndexFlatIP(d)
index_flat.add(catalog_embeddings)
print(f'IndexFlatIP built: {index_flat.ntotal:,} vectors')

# 2. IVF index — approximate search, used for fast deployment queries
nlist = 200  # number of clusters; rule of thumb is sqrt(n) to 4*sqrt(n)
quantizer = faiss.IndexFlatIP(d)
index_ivf = faiss.IndexIVFFlat(quantizer, d, nlist, faiss.METRIC_INNER_PRODUCT)

print(f'Training IVF index with nlist={nlist}...')
index_ivf.train(catalog_embeddings)
index_ivf.add(catalog_embeddings)
index_ivf.nprobe = 10  # number of clusters searched per query (speed/accuracy tradeoff)
print(f'IndexIVFFlat built and trained: {index_ivf.ntotal:,} vectors | nprobe={index_ivf.nprobe}')

# Save both indexes to disk
faiss.write_index(index_flat, os.path.join(INDEX_DIR, 'sop_flat.index'))
faiss.write_index(index_ivf,  os.path.join(INDEX_DIR, 'sop_ivf.index'))
print(f'\nIndexes saved to {INDEX_DIR}')

Building FAISS indexes for 120,053 vectors of dimension 512
IndexFlatIP built: 120,053 vectors
Training IVF index with nlist=200...
IndexIVFFlat built and trained: 120,053 vectors | nprobe=10

Indexes saved to C:\visual_searh_project\faiss_index


## Cell 6 — Latency Benchmark: Brute-Force vs FAISS Flat vs FAISS IVF

In [22]:
# Extract query embeddings (test set) to benchmark search speed and accuracy.
print('Extracting query embeddings (test set)...')
query_embeddings, query_labels, _ = extract_embeddings(test_loader, model, DEVICE, desc='Queries')

N_BENCHMARK_QUERIES = 500
bench_idx = np.random.RandomState(42).choice(len(query_embeddings), N_BENCHMARK_QUERIES, replace=False)
bench_queries = query_embeddings[bench_idx]

def benchmark_search(search_fn, queries, k=10, n_runs=1):
    search_fn(queries[:5], k)  # warm-up, excluded from timing
    t0 = time.time()
    for _ in range(n_runs):
        search_fn(queries, k)
    elapsed = time.time() - t0
    per_query_ms = (elapsed / n_runs / len(queries)) * 1000
    return per_query_ms

def brute_force_search(q, k):
    sims = q @ catalog_embeddings.T
    return np.argsort(-sims, axis=1)[:, :k]

def flat_search(q, k):
    _, idx = index_flat.search(q, k)
    return idx

def ivf_search(q, k):
    _, idx = index_ivf.search(q, k)
    return idx

print(f'\nBenchmarking on {N_BENCHMARK_QUERIES} queries against {n:,} catalog vectors, K=10')
print('-' * 60)

bf_ms   = benchmark_search(brute_force_search, bench_queries, k=10)
flat_ms = benchmark_search(flat_search,        bench_queries, k=10)
ivf_ms  = benchmark_search(ivf_search,         bench_queries, k=10)

print(f'{"Method":<28}| {"Latency / query":>18}')
print('-' * 60)
print(f'{"NumPy brute-force":<28}| {bf_ms:>14.3f} ms')
print(f'{"FAISS IndexFlatIP":<28}| {flat_ms:>14.3f} ms')
print(f'{"FAISS IndexIVFFlat":<28}| {ivf_ms:>14.3f} ms')
print('-' * 60)
print(f'IVF speedup vs brute-force : {bf_ms/ivf_ms:.1f}x faster')
print(f'IVF speedup vs FAISS Flat  : {flat_ms/ivf_ms:.1f}x faster')

Extracting query embeddings (test set)...
  Queries: 6,400/60,502...
  Queries: 12,800/60,502...
  Queries: 19,200/60,502...
  Queries: 25,600/60,502...
  Queries: 32,000/60,502...
  Queries: 38,400/60,502...
  Queries: 44,800/60,502...
  Queries: 51,200/60,502...
  Queries: 57,600/60,502...
Queries complete in 25.8 min

Benchmarking on 500 queries against 120,053 catalog vectors, K=10
------------------------------------------------------------
Method                      |    Latency / query
------------------------------------------------------------
NumPy brute-force           |          2.505 ms
FAISS IndexFlatIP           |          0.392 ms
FAISS IndexIVFFlat          |          0.288 ms
------------------------------------------------------------
IVF speedup vs brute-force : 8.7x faster
IVF speedup vs FAISS Flat  : 1.4x faster


## Cell 7 — Recall@K: Brute-Force vs FAISS Flat vs FAISS IVF (Accuracy Check)

In [23]:
# Critical check: does the FAISS IVF approximate index lose retrieval accuracy
# compared to exact search? We compare Recall@10 for both methods on the full
# test set, evaluated against the full catalog.
#
# Since test images ARE part of the catalog, every query will trivially match
# itself (similarity = 1.0). We exclude the exact self-match by catalog INDEX
# (not by assuming it's always rank 0 — ties or near-duplicates could break
# a position-based assumption). We build a path -> catalog_index lookup once,
# then explicitly filter out that exact index from each query's results before
# checking Recall@10. This is the same self-exclusion principle used in
# Phase 2/3 evaluation (mask diagonal), applied here via FAISS result filtering.

catalog_paths_arr = np.load(os.path.join(INDEX_DIR, 'catalog_paths.npy'), allow_pickle=True)
path_to_catalog_idx = {p: i for i, p in enumerate(catalog_paths_arr)}
query_self_idx = np.array([path_to_catalog_idx[p] for p, _ in test_samples])

K_EVAL = 15  # search a few extra so we still have 10 results after dropping self-match

def recall_excluding_self(index, query_emb, query_lbl, query_self_idx, catalog_lbl, k=10):
    # Vectorized: search k+1 to handle self-match, then filter with numpy
    _, idx = index.search(query_emb, k + 1)                    # (Q, k+1)
    self_col = query_self_idx.reshape(-1, 1)                   # (Q, 1)
    mask     = idx != self_col                                  # (Q, k+1) bool
    # Take first k True positions per row
    filtered = np.array([row[m][:k] for row, m in zip(idx, mask)])  # (Q, k)
    retrieved_labels = catalog_lbl[filtered]                   # (Q, k)
    correct = (retrieved_labels == query_lbl.reshape(-1, 1)).any(axis=1)
    return correct.mean()

print('Evaluating Recall@10 for FAISS Flat (exact) vs FAISS IVF (approximate)...')
print('Self-matches excluded by catalog index, not by assumed rank position.')
print('-' * 60)

r10_flat = recall_excluding_self(index_flat, query_embeddings, query_labels,
                                  query_self_idx, catalog_labels, k=10)
r10_ivf  = recall_excluding_self(index_ivf, query_embeddings, query_labels,
                                  query_self_idx, catalog_labels, k=10)

print(f'{"Method":<28}| {"Recall@10":>12}')
print('-' * 60)
print(f'{"FAISS IndexFlatIP (exact)":<28}| {r10_flat*100:>10.2f}%')
print(f'{"FAISS IndexIVFFlat (ANN)":<28}| {r10_ivf*100:>10.2f}%')
print('-' * 60)
print(f'Accuracy drop from approximation: {(r10_flat - r10_ivf)*100:.2f} percentage points')

Evaluating Recall@10 for FAISS Flat (exact) vs FAISS IVF (approximate)...
Self-matches excluded by catalog index, not by assumed rank position.
------------------------------------------------------------
Method                      |    Recall@10
------------------------------------------------------------
FAISS IndexFlatIP (exact)   |      84.50%
FAISS IndexIVFFlat (ANN)    |      81.94%
------------------------------------------------------------
Accuracy drop from approximation: 2.56 percentage points


## Cell 7b — nprobe Sweep: Speed vs Accuracy Tradeoff (IVF)

In [24]:
# Sweep nprobe from 1 to 50 to find the optimal speed/accuracy tradeoff.
# More nprobe = more clusters searched = higher recall but slower.
# This plot is useful for your colloquium: shows how to tune the ANN index.

nprobe_values = [1, 2, 5, 10, 20, 30, 50]
nprobe_results = []

# Use a subset for speed (1000 queries is enough for stable estimates)
sweep_idx = np.random.RandomState(0).choice(len(query_embeddings), 1000, replace=False)
sweep_q   = query_embeddings[sweep_idx]
sweep_lbl = query_labels[sweep_idx]
sweep_self = query_self_idx[sweep_idx]

print(f'{"nprobe":<10} | {"Recall@10":>10} | {"Latency (ms)":>14}')
print('-' * 40)

for np_ in nprobe_values:
    index_ivf.nprobe = np_
    # Recall
    r10 = recall_excluding_self(index_ivf, sweep_q, sweep_lbl, sweep_self, catalog_labels, k=10)
    # Latency
    index_ivf.search(sweep_q[:5], 10)  # warm-up
    t0 = time.time()
    index_ivf.search(sweep_q, 10)
    ms = (time.time() - t0) / len(sweep_q) * 1000
    nprobe_results.append({'nprobe': np_, 'recall10': r10, 'latency_ms': ms})
    print(f'{np_:<10} | {r10*100:>9.2f}% | {ms:>12.3f} ms')

# Restore a good default
best_nprobe = max(nprobe_results, key=lambda x: x['recall10'] / (x['latency_ms'] + 1e-9))['nprobe']
index_ivf.nprobe = best_nprobe
print(f'\nBest nprobe by recall/latency ratio: {best_nprobe}')
print(f'Re-saving IVF index with nprobe={best_nprobe}...')
faiss.write_index(index_ivf, os.path.join(INDEX_DIR, 'sop_ivf.index'))
print('Done.')

nprobe     |  Recall@10 |   Latency (ms)
----------------------------------------
1          |     63.20% |        0.031 ms
2          |     72.10% |        0.060 ms
5          |     78.90% |        0.152 ms
10         |     80.70% |        0.311 ms
20         |     82.30% |        0.573 ms
30         |     82.90% |        0.787 ms
50         |     83.50% |        1.317 ms

Best nprobe by recall/latency ratio: 1
Re-saving IVF index with nprobe=1...
Done.


## Cell 7c — HNSW Index: Even Faster ANN Search

In [25]:
# HNSW (Hierarchical Navigable Small World) is often faster than IVF
# at high recall. No training needed, memory-resident only.
# M=32: neighbors per node (higher = more accurate, more memory)
# efSearch=64: beam width at query time (higher = more accurate, slower)

M = 32
index_hnsw = faiss.IndexHNSWFlat(d, M, faiss.METRIC_INNER_PRODUCT)
index_hnsw.hnsw.efConstruction = 200  # build quality (only affects build time)

print(f'Building HNSW index (M={M}, efConstruction=200)...')
t0 = time.time()
index_hnsw.add(catalog_embeddings)
print(f'HNSW built in {(time.time()-t0)/60:.1f} min | {index_hnsw.ntotal:,} vectors')

# Sweep efSearch
ef_values = [16, 32, 64, 128]
print(f'\n{"efSearch":<10} | {"Recall@10":>10} | {"Latency (ms)":>14}')
print('-' * 40)
hnsw_results = []
for ef in ef_values:
    index_hnsw.hnsw.efSearch = ef
    r10 = recall_excluding_self(index_hnsw, sweep_q, sweep_lbl, sweep_self, catalog_labels, k=10)
    index_hnsw.search(sweep_q[:5], 10)
    t0 = time.time()
    index_hnsw.search(sweep_q, 10)
    ms = (time.time()-t0)/len(sweep_q)*1000
    hnsw_results.append({'efSearch':ef,'recall10':r10,'latency_ms':ms})
    print(f'{ef:<10} | {r10*100:>9.2f}% | {ms:>12.3f} ms')

# Save HNSW
index_hnsw.hnsw.efSearch = 64  # good default
faiss.write_index(index_hnsw, os.path.join(INDEX_DIR, 'sop_hnsw.index'))
print(f'\nHNSW index saved.')

# Three-way comparison summary
best_ivf  = next(r for r in nprobe_results if r['nprobe'] == index_ivf.nprobe)
best_hnsw = next(r for r in hnsw_results  if r['efSearch'] == 64)
print(f'\n=== Three-way ANN Comparison (1000 queries) ===')
print(f'  IVF  (nprobe={index_ivf.nprobe})   : R@10={best_ivf["recall10"]*100:.2f}% | {best_ivf["latency_ms"]:.3f} ms')
print(f'  HNSW (efSearch=64) : R@10={best_hnsw["recall10"]*100:.2f}% | {best_hnsw["latency_ms"]:.3f} ms')

Building HNSW index (M=32, efConstruction=200)...
HNSW built in 1.0 min | 120,053 vectors

efSearch   |  Recall@10 |   Latency (ms)
----------------------------------------
16         |     83.80% |        0.060 ms
32         |     83.80% |        0.116 ms
64         |     83.70% |        0.185 ms
128        |     83.70% |        0.331 ms

HNSW index saved.

=== Three-way ANN Comparison (1000 queries) ===
  IVF  (nprobe=1)   : R@10=63.20% | 0.031 ms
  HNSW (efSearch=64) : R@10=83.70% | 0.185 ms


## Cell 8 — End-to-End Query Function (Image In, Top-K Results Out)

In [26]:
# This is the function the Streamlit app (Phase 5) will call directly.

def search_by_image(image_path, model, index, catalog_paths, catalog_labels,
                     transform, device, k=5):
    
    
  
    
    t0 = time.time()
    img = Image.open(image_path).convert('RGB')
    img_t = transform(img).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            query_emb = model(img_t).cpu().numpy().astype('float32')

    sims, idxs = index.search(query_emb, k)
    latency_ms = (time.time() - t0) * 1000

    results = []
    for rank in range(k):
        idx = idxs[0][rank]
        results.append({
            'rank': rank + 1,
            'path': catalog_paths[idx],
            'label': int(catalog_labels[idx]),
            'similarity': float(sims[0][rank])
        })
    return results, latency_ms

# Quick functional test with a random test-set image
sample_path = test_samples[0][0]
results, latency = search_by_image(
    sample_path, model, index_ivf, catalog_paths=np.load(os.path.join(INDEX_DIR, 'catalog_paths.npy'), allow_pickle=True),
    catalog_labels=catalog_labels, transform=TEST_TF, device=DEVICE, k=5
)

print(f'Query image: {sample_path}')
print(f'Latency: {latency:.2f} ms')
print(f'\nTop-5 results:')
for r in results:
    print(f'  Rank {r["rank"]} | Label {r["label"]} | Similarity {r["similarity"]:.4f} | {os.path.basename(r["path"])}')

Query image: C:\visual_searh_project\data\raw_data\Stanford_Online_Products\Stanford_Online_Products\bicycle_final/251952414262_0.JPG
Latency: 179.91 ms

Top-5 results:
  Rank 1 | Label 11319 | Similarity 1.0000 | 251952414262_0.JPG
  Rank 2 | Label 11400 | Similarity 0.6434 | 261930777111_2.JPG
  Rank 3 | Label 11400 | Similarity 0.5537 | 261930777111_1.JPG
  Rank 4 | Label 11481 | Similarity 0.5448 | 271933533351_0.JPG
  Rank 5 | Label 11532 | Similarity 0.5355 | 281666266667_1.JPG


## Cell 9 — Summary Table & Save Final Artifacts for Phase 5

In [27]:
print('='*65)
print('  PHASE 4 SUMMARY — FAISS Integration')
print('='*65)
print(f'  Catalog size              : {n:,} images')
print(f'  Embedding dimension       : {d}')
print()
print(f'  {"Metric":<28}| {"Brute-Force":>12}| {"FAISS Flat":>12}| {"FAISS IVF":>12}')
print('  ' + '-'*61)
print(f'  {"Latency / query (ms)":<28}| {bf_ms:>10.3f}  | {flat_ms:>10.3f}  | {ivf_ms:>10.3f}')
print(f'  {"Recall@10":<28}| {"—":>12}| {r10_flat*100:>10.2f}% | {r10_ivf*100:>10.2f}%')
print('='*65)
print(f'  Speedup (IVF vs brute-force): {bf_ms/ivf_ms:.1f}x')
print(f'  Recall lost to approximation: {(r10_flat-r10_ivf)*100:.2f} percentage points')
print('='*65)

# Save a summary JSON for the colloquium report and README
summary = {
    'catalog_size': int(n),
    'embedding_dim': int(d),
    'nlist': int(nlist),
    'nprobe': int(index_ivf.nprobe),
    'latency_ms': {
        'brute_force': float(bf_ms),
        'faiss_flat': float(flat_ms),
        'faiss_ivf': float(ivf_ms),
    },
    'recall_at_10': {
        'faiss_flat_exact': float(r10_flat),
        'faiss_ivf_approx': float(r10_ivf),
    },
    'speedup_ivf_vs_bruteforce': float(bf_ms / ivf_ms),
}

SUMMARY_PATH = os.path.join(INDEX_DIR, 'phase4_summary.json')
with open(SUMMARY_PATH, 'w') as f:
    json.dump(summary, f, indent=2)

print(f'\nSaved summary to {SUMMARY_PATH}')
print(f'Saved FAISS indexes to {INDEX_DIR}\\sop_flat.index and {INDEX_DIR}\\sop_ivf.index')
print(f'Saved catalog embeddings/labels/paths to {INDEX_DIR}')
print('\nPhase 4 complete. All artifacts ready for Phase 5 (Streamlit deployment).')

  PHASE 4 SUMMARY — FAISS Integration
  Catalog size              : 120,053 images
  Embedding dimension       : 512

  Metric                      |  Brute-Force|   FAISS Flat|    FAISS IVF
  -------------------------------------------------------------
  Latency / query (ms)        |      2.505  |      0.392  |      0.288
  Recall@10                   |            —|      84.50% |      81.94%
  Speedup (IVF vs brute-force): 8.7x
  Recall lost to approximation: 2.56 percentage points

Saved summary to C:\visual_searh_project\faiss_index\phase4_summary.json
Saved FAISS indexes to C:\visual_searh_project\faiss_index\sop_flat.index and C:\visual_searh_project\faiss_index\sop_ivf.index
Saved catalog embeddings/labels/paths to C:\visual_searh_project\faiss_index

Phase 4 complete. All artifacts ready for Phase 5 (Streamlit deployment).
